In [66]:
import pandas as pd
df = pd.read_csv("./dataset/work6.csv")
df

,text
0,อาหารหวานมาก ไม่อร่อย
1,กินอาหารอร่อยมาก หวานดี อร่อยดี
2,อาหารอร่อยดีไม่หวาน


In [67]:
import nltk
from pythainlp.corpus.common import thai_stopwords
from pythainlp.tokenize import word_tokenize
import re
import string
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt_tab to /home/night/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [68]:
import pythainlp
from pythainlp.util import emoji_to_thai


#function for removing number
def remove_numbers(text):
    number_pattern = r'\d+'
    without_number = re.sub(pattern=number_pattern, repl=" ", string=text)
    return without_number

#function for removing punctuation
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

#function for removing stopword
def remove_stopwords(text):
    removed = []
    #stop_words = list(stopwords.words("english"))
    stopwords = list(thai_stopwords())
    tokens = word_tokenize(text)
    for i in range(len(tokens)):
        if tokens[i] not in stopwords:
            removed.append(tokens[i])
    return "".join(removed)

def tokenization_thai(text):
    token = word_tokenize(text,engine='longest', keep_whitespace=False)
    return " ".join(token)


#function for removing extra white_spaces
def remove_extra_white_spaces(text):
    space_pattern = r'\s+'
    without_sc = re.sub(pattern=space_pattern, repl="", string=text)
    return without_sc



In [69]:
df['text'] = df['text'].apply(lambda x: remove_numbers(x)) #remove number
df['text'] = df['text'].apply(lambda x: remove_punctuation(x)) #remove punc
df['text'] = df['text'].apply(lambda x: remove_stopwords(x)) #remove stopword

df['text'] = df['text'].apply(lambda x: tokenization_thai(x)) #Lemmatization
df['text'] = df['text'].apply(lambda x: remove_extra_white_spaces(x)) #remove white space
df.head()

,text
0,อาหารหวานอร่อย
1,กินอาหารอร่อยหวานดีอร่อยดี
2,อาหารอร่อยดีหวาน


# onehot

In [70]:
vocab = set()
for text in df['text']:
    tokens = word_tokenize(text)
    for token in tokens:
        vocab.add(token)
vocab = list(vocab)
vocab

['ดี', 'อาหาร', 'อร่อย', 'หวาน', 'กิน', 'อาหารหวาน']

In [71]:
#create one hot vector of dataset
import numpy as np

# Create a dictionary to map words to integers
word_to_int = {word: i for i, word in enumerate(vocab)}
print(word_to_int)

# Create a binary vector for each word in each sentence
one_hot_vectors = []
for sentence in df['text']:
    words = word_tokenize(sentence)
    sentence_vectors = []
    for word in words:
        binary_vector = np.zeros(len(vocab)) #ให้ vector เป็น 0 หมดก่อน [0,0,0,0]
        binary_vector[word_to_int[word]] = 1 #first [0,0,1,0]
        sentence_vectors.append(binary_vector)
    one_hot_vectors.append(sentence_vectors)

one_hot_vectors

{'ดี': 0, 'อาหาร': 1, 'อร่อย': 2, 'หวาน': 3, 'กิน': 4, 'อาหารหวาน': 5}


[[array([0., 0., 0., 0., 0., 1.]), array([0., 0., 1., 0., 0., 0.])],
 [array([0., 0., 0., 0., 1., 0.]),
  array([0., 1., 0., 0., 0., 0.]),
  array([0., 0., 1., 0., 0., 0.]),
  array([0., 0., 0., 1., 0., 0.]),
  array([1., 0., 0., 0., 0., 0.]),
  array([0., 0., 1., 0., 0., 0.]),
  array([1., 0., 0., 0., 0., 0.])],
 [array([0., 1., 0., 0., 0., 0.]),
  array([0., 0., 1., 0., 0., 0.]),
  array([1., 0., 0., 0., 0., 0.]),
  array([0., 0., 0., 1., 0., 0.])]]

In [72]:
#find max len of documents for padding
max_sent = max(one_hot_vectors , key = len)
max_sent_length = len(max_sent)
max_sent_length

7

In [73]:
#pad = post to the same size for all documents
dimension = len(vocab)
pad = np.zeros(shape=(1, dimension)) #[0,0,0,0]
X=[]
for x in one_hot_vectors:
  if len(x)<max_sent_length:
    dif_len = max_sent_length-len(x)
    for i in range(0,dif_len):
      x = np.concatenate([x, pad])
  X.append(x)
X = np.array(X)
print(X)
print(X.shape)

[[[0. 0. 0. 0. 0. 1.]
  [0. 0. 1. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 1. 0.]
  [0. 1. 0. 0. 0. 0.]
  [0. 0. 1. 0. 0. 0.]
  [0. 0. 0. 1. 0. 0.]
  [1. 0. 0. 0. 0. 0.]
  [0. 0. 1. 0. 0. 0.]
  [1. 0. 0. 0. 0. 0.]]

 [[0. 1. 0. 0. 0. 0.]
  [0. 0. 1. 0. 0. 0.]
  [1. 0. 0. 0. 0. 0.]
  [0. 0. 0. 1. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]]]
(3, 7, 6)


# wordEmbedding

In [ ]:
doc_tokens = [] 
for text in df['text']:
    tokens = word_tokenize(text)
    doc_tokens.append(tokens)

doc_tokens

[['อาหารหวาน', 'อร่อย'],
 ['กิน', 'อาหาร', 'อร่อย', 'หวาน', 'ดี', 'อร่อย', 'ดี'],
 ['อาหาร', 'อร่อย', 'ดี', 'หวาน']]

In [75]:
# without pretrained model, train model from your dataset
from pythainlp import word_vector

# train model
#important parameters
# vector_size: (default 100) The number of dimensions of the embedding
# sg: (default 0 or CBOW) The training algorithm, either CBOW (0) or skip gram (1).
# min_count: (default 5) The minimum count of words to consider when training the model; words with an occurrence less than this count will be ignored.



dimension = 10 #define number of dimension of word embeding vector
model = word_vector.WordVector(model_name="thai2fit_wv").get_model()





Corpus: thai2fit_wv
- Downloading: thai2fit_wv 0.1


/home/night/Git_Hub/-natural-language-processing/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 62452646/62452646 [00:05<00:00, 12079662.10it/s]


In [76]:
def word_vectors(text, word_dict=model.index_to_key):
  tokens = word_tokenize(text)

  vectors = []

  for token in tokens:
    if token not in word_dict:
      continue
    token_vector = model[token]
    vectors.append(token_vector)
  return vectors

In [77]:
# load model that was created from your dataset
text ="หวาน"
embed_vector = word_vectors(text)
print(embed_vector)



[array([ 1.59253e-01,  6.43510e-02, -1.76604e-01, -1.21169e-01,
       -6.12690e-02, -3.63560e-02, -3.05810e-02, -2.06242e-01,
       -1.73866e-01, -1.41345e-01, -1.41503e-01, -1.75380e-02,
       -4.11740e-02,  5.81800e-03,  5.09390e-02,  2.70040e-02,
       -1.15591e-01,  5.46000e-03, -1.85452e-01, -9.02610e-02,
       -2.49716e-01, -4.21500e-03,  1.11350e-01, -2.42830e-02,
       -3.94520e-02, -6.49660e-02, -1.76680e-01, -1.18322e-01,
       -6.19290e-02,  1.02896e-01, -1.04870e-01, -1.89652e-01,
        5.50480e-02, -2.96387e-01,  6.47680e-02, -8.37700e-02,
       -3.55800e-01,  7.02300e-02,  4.28880e-02, -6.12140e-02,
        1.18126e-01,  3.37119e-01, -2.49170e-02, -4.82070e-02,
        8.26140e-02, -6.36990e-02,  2.24728e-01,  2.95293e-01,
        9.75270e-02,  5.14034e-01, -4.19231e-01,  6.84220e-02,
       -7.21400e-02,  1.29313e-01,  7.03060e-02, -8.16520e-02,
        7.12250e-02,  1.46544e-01,  3.92140e-01, -6.48500e-02,
        1.31805e-01, -1.22907e-01, -4.21550e-02, -7.90